In [ ]:
import sys, warnings
from pathlib import Path
import torch

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences, heterogeneous_sampler

warnings.filterwarnings("ignore")
print("imports OK")

## Test — `fork_sequences` unit tests

In [ ]:
print("=" * 60)
print("TEST 8 — fork_sequences shape assertions")
print("=" * 60)

def make_batch(B=2, T=200, C=3, Vh=1, H=6, device="cpu"):
    """Minimal collated batch as _full_series_collate_fn would produce."""
    return {
        "x_enc":          torch.randn(B, T, C, 1 + Vh, device=device),
        "available_mask": torch.ones(B, T, C, device=device),   # [B, T, C]
        "channel_mask":   torch.ones(B, C, device=device),
        "horizon":        torch.full((B,), H, dtype=torch.long, device=device),
    }

L, H, S = 32, 6, 1

# ── Training path (fcd_samples=4) ─────────────────────────────────────────────
print("\n--- Training path (fcd_samples=4) ---")
batch = make_batch(B=2, T=200, C=3, Vh=1, H=H)
out   = fork_sequences(batch, context_length=L, fcd_samples=4, horizon=H)
B_out, enc_size, C_out, feat = out["insample_y"].shape
print(f"insample_y:    {tuple(out['insample_y'].shape)}    expected [2, enc_size, 3, 2]")
print(f"outsample_y:   {tuple(out['outsample_y'].shape)}   expected [2, 4, {H}, 3]")
print(f"available_mask:{tuple(out['available_mask'].shape)}")
assert out["outsample_y"].shape == (2, 4, H, 3), \
    f"outsample_y shape wrong: {out['outsample_y'].shape}"
assert out["available_mask"].shape == (2, enc_size, 3), \
    f"available_mask shape wrong: {out['available_mask'].shape}"
print("training path shapes ✓")

# ── Val/test path (fcd_samples=-1) ────────────────────────────────────────────
print("\n--- Val/test path (fcd_samples=-1) ---")
T = 200
batch_val  = make_batch(B=2, T=T, C=3, Vh=0, H=H)
out_val    = fork_sequences(batch_val, context_length=L, fcd_samples=-1, horizon=H)
n_expected = T - L - H + 1
print(f"n_valid_fcds({T}, L={L}, H={H}, S={S}) = {n_expected}")
print(f"outsample_y:   {tuple(out_val['outsample_y'].shape)}  expected [2, {n_expected}, {H}, 3]")
assert out_val["outsample_y"].shape[1] == n_expected, \
    f"n_fcds wrong: {out_val['outsample_y'].shape[1]} != {n_expected}"
print("val/test path shapes ✓")

# ── Left-padding: windows never start in padding ───────────────────────────────
print("\n--- Left-padding: sampler skips padded timesteps ---")
T_real, T_pad = 100, 50
T_total       = T_real + T_pad
mask = torch.zeros(1, T_total, 3)       # [B, T, C]
mask[:, T_pad:, :] = 1.0               # real data starts at T_pad
batch_pad = {
    "x_enc":          torch.randn(1, T_total, 3, 1),
    "available_mask": mask,
    "horizon":        torch.tensor([H]),
}

for _ in range(20):
    ws, _ = heterogeneous_sampler(mask, L, fcd_samples=4, horizon=H)
    assert ws[0].item() >= T_pad, \
        f"window_start {ws[0].item()} < T_pad {T_pad} — sampled in padding!"
print("all 20 sampled window_starts ≥ T_pad ✓")

# ── available_mask shape assertion ────────────────────────────────────────────
print("\n--- available_mask shape mismatch raises AssertionError ---")
bad_mask_batch = {
    "x_enc":          torch.randn(2, 100, 3, 1),
    "available_mask": torch.ones(2, 3, 100),   # wrong: [B, C, T] instead of [B, T, C]
    "horizon":        torch.tensor([H, H]),
}
try:
    fork_sequences(bad_mask_batch, context_length=L, fcd_samples=4, horizon=H)
    assert False, "should have raised"
except AssertionError as e:
    print(f"raised AssertionError as expected: {e}")

print("\n✓ TEST PASSED")